# Music Annotation Project
## Notebook 72R v2: Post-Lock Randomised Three-Way Evaluation

**Runtime: CPU.** Run only after Notebook 72 has closed its prediction lock.

This revision fixes the Gradio save error and changes the review instrument. It does not ask for a forced “best” page. Instead, it records the three core judgements independently for **each** concealed version A, B and C:

1. Is the page playable from?
2. Is printed notation visibly damaged or destroyed?
3. Has annotation been removed?

A separate preference field records ordering only. Ties are allowed. Preference is not treated as a safety or usability verdict.

The concealed variants are:
- observed annotated control;
- frozen V7.8 baseline;
- unified V8.8 corrected page.

The Gradio error `'< not supported between instances of int and str'` was caused by mixed numeric/string values in persisted review-order data. This version normalises `review_order` to pandas nullable integer and sorts only on that numeric field.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" opencv-python-headless pandas numpy pillow

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, hmac, json, platform, secrets, shutil, sys, zipfile
import cv2, gradio as gr, numpy as np, pandas as pd

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
SOURCE_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook72r_post_lock_review'
REVIEWER_CODE='DB'
REVIEW_REVISION='72R-v2-three-way-independent-judgements'
RANDOM_SEED=720088
PUBLIC_LINK=True
REQUIRE_ALL_151=True

REQUIRED_SOURCE=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','tables/runtime_page_summary.csv','pages']
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NOTEBOOK72_ROOT,REQUIRED_SOURCE)
if SOURCE is None: raise RuntimeError('No complete Notebook 72 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72r_v2_three_way_review'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; REVIEW=OUT/'review'; SEALED=OUT/'sealed'; DECODED=OUT/'decoded'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,REVIEW,SEALED,DECODED,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Source Notebook 72:',SOURCE)
print('Review output:',OUT)
print('Gradio:',gr.__version__)

## 2. Integrity helpers and Notebook 72 lock verification

In [ ]:
def require(ok,msg):
    if not bool(ok): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp'); t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); t.replace(path)
def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unreadable image: {path}'); return a

lock=json.loads((SOURCE/'prediction_lock.json').read_text()); verdict=json.loads((SOURCE/'notebook72_verdict.json').read_text())
require(lock.get('state')=='CLOSED','Notebook 72 prediction lock is not CLOSED.')
require(verdict.get('first_failed_gate') is None,'Notebook 72 has a failed gate.')
roster=pd.read_csv(SOURCE/'inputs/runtime_roster.csv',dtype=str).fillna('')
hashes=pd.read_csv(SOURCE/'tables/page_output_hashes.csv',dtype=str).fillna('').set_index('page_key')
summary=pd.read_csv(SOURCE/'tables/runtime_page_summary.csv')
require(len(roster)==len(hashes)==len(summary),'Notebook 72 roster/hash/summary counts differ.')
if REQUIRE_ALL_151: require(len(roster)==151,f'Expected 151 pages; found {len(roster)}.')
require(lock['payload']['runtime_roster_sha256']==sha256_file(SOURCE/'inputs/runtime_roster.csv'),'Roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(SOURCE/'tables/page_output_hashes.csv'),'Output hashes changed after lock.')
source_identity={'source_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'operator_bundle_id':lock['payload']['operator_bundle_id'],'pages':len(roster),'review_revision':REVIEW_REVISION,'reviewer_code':REVIEWER_CODE,'gradio_version':gr.__version__}
atomic_json(INPUTS/'source_identity.json',source_identity)
print(json.dumps(source_identity,indent=2))

## 3. Verify all three locked variants

In [ ]:
records=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; d=SOURCE/'pages'/key
    observed=read_gray(d/'observed.png'); baseline=read_gray(d/'baseline_v78.png'); corrected=read_gray(d/'corrected.png')
    require(observed.shape==baseline.shape==corrected.shape,f'{key}: shape mismatch.')
    h=hashes.loc[key]
    require(array_hash(observed)==h.observed_array_sha256,f'{key}: observed hash mismatch.')
    require(array_hash(baseline)==h.baseline_array_sha256,f'{key}: baseline hash mismatch.')
    require(array_hash(corrected)==h.corrected_array_sha256,f'{key}: corrected hash mismatch.')
    records.append({'page_key':key,'page':r.page,'population':r.population,'observed_path':str(d/'observed.png'),'baseline_path':str(d/'baseline_v78.png'),'corrected_path':str(d/'corrected.png')})
pages=pd.DataFrame(records); pages.to_csv(INPUTS/'verified_review_roster.csv',index=False)
print('Verified:',len(pages),'pages x 3 variants')

## 4. Create and seal per-page A/B/C assignments

In [ ]:
secret=secrets.token_bytes(32); rng=np.random.default_rng(RANDOM_SEED); rows=[]
for order,(_,r) in enumerate(pages.iterrows(),1):
    perm=list(rng.permutation(['OBSERVED','BASELINE_V78','CORRECTED_V88']))
    row={'review_order':int(order),'page_key':str(r.page_key),'page':str(r.page),'population':str(r.population),'A':str(perm[0]),'B':str(perm[1]),'C':str(perm[2])}
    row['assignment_commitment']=hmac.new(secret,json.dumps(row,sort_keys=True).encode(),hashlib.sha256).hexdigest(); rows.append(row)
key=pd.DataFrame(rows); key['review_order']=pd.to_numeric(key.review_order,errors='raise').astype('Int64'); key.to_csv(SEALED/'review_key.csv',index=False)
atomic_json(SEALED/'review_key_seal.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'review_key_sha256':sha256_file(SEALED/'review_key.csv'),'secret_b64':base64.b64encode(secret).decode(),'status':'SEALED_UNTIL_REVIEW_LOCK'})
public=key[['review_order','page_key','assignment_commitment']].copy(); public.to_csv(REVIEW/'review_queue_public.csv',index=False)
print('Review queue:',len(public))

## 5. Typed persistent response store

In [ ]:
RESPONSES=REVIEW/'responses.csv'; HISTORY=REVIEW/'response_history.jsonl'; REVIEW_LOCK=OUT/'review_lock.json'
VARIANTS=['A','B','C']
PER_VARIANT_FIELDS=['playable','print_damage','annotation_removal','residue','variant_notes']
GENERAL_FIELDS=['preference_order','confidence','overall_notes']
RESPONSE_COLUMNS=['review_order','page_key','reviewer_code','review_revision','started_utc','submitted_utc']+[f'{v}_{f}' for v in VARIANTS for f in PER_VARIANT_FIELDS]+GENERAL_FIELDS+['assignment_commitment','response_sha256']
if not RESPONSES.exists(): pd.DataFrame(columns=RESPONSE_COLUMNS).to_csv(RESPONSES,index=False)

def response_frame():
    q=pd.read_csv(RESPONSES,dtype=str,keep_default_na=False)
    for c in RESPONSE_COLUMNS:
        if c not in q.columns: q[c]=''
    q=q[RESPONSE_COLUMNS].copy()
    q['review_order']=pd.to_numeric(q['review_order'],errors='coerce').astype('Int64')
    require(not q.page_key.duplicated().any(),'Duplicate current response rows.')
    return q

def response_hash(row): return object_hash({k:('' if pd.isna(row.get(k,'')) else row.get(k,'')) for k in RESPONSE_COLUMNS if k!='response_sha256'})
def completed_keys(): return set(response_frame().page_key.astype(str))
def review_closed(): return REVIEW_LOCK.exists()
def save_response(row):
    require(not review_closed(),'Review is locked.')
    row={k:row.get(k,'') for k in RESPONSE_COLUMNS}; row['review_order']=int(row['review_order']); row['response_sha256']=response_hash(row)
    q=response_frame(); old=q[q.page_key==row['page_key']]
    with open(HISTORY,'a') as f: f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'event':'create' if old.empty else 'replace','page_key':row['page_key'],'old':None if old.empty else old.iloc[0].astype(str).to_dict(),'new':row},sort_keys=True)+'\n')
    q=q[q.page_key!=row['page_key']]; q=pd.concat([q,pd.DataFrame([row])],ignore_index=True); q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype('Int64'); q=q.sort_values(['review_order','page_key'],kind='stable'); q.to_csv(RESPONSES,index=False)
    return row['response_sha256']
print('Typed response store ready.')

## 6. Launch revised reviewer

In [ ]:
LOOKUP=pages.set_index('page_key').to_dict('index'); KEY=key.set_index('page_key').to_dict('index'); PUBLIC=public.set_index('page_key').to_dict('index')
def variant_path(page_key,variant):
    r=LOOKUP[page_key]; return Path({'OBSERVED':r['observed_path'],'BASELINE_V78':r['baseline_path'],'CORRECTED_V88':r['corrected_path']}[variant])
def render(page_key,label): return read_gray(variant_path(page_key,KEY[page_key][label]))
def pending():
    done=completed_keys(); q=public[~public.page_key.astype(str).isin(done)].copy(); q['review_order']=pd.to_numeric(q.review_order,errors='raise').astype('Int64'); return q.sort_values(['review_order','page_key'],kind='stable').page_key.astype(str).tolist()
def status_text(): return f'Completed {len(completed_keys())} / {len(public)} | Remaining {len(public)-len(completed_keys())}'

def defaults():
    x=[]
    for _ in VARIANTS: x += ['UNCERTAIN','UNCERTAIN','PARTIAL','VISIBLE_RESIDUE','']
    return x+['A=B=C','MEDIUM','']
def load_page(page_key):
    if not page_key: return [None,None,None,status_text()]+defaults()
    return [render(page_key,'A'),render(page_key,'B'),render(page_key,'C'),f'{status_text()} | item {int(PUBLIC[page_key]["review_order"])}']+defaults()

def submit(page_key,*values):
    if not page_key: raise gr.Error('Select a page.')
    names=[f'{v}_{f}' for v in VARIANTS for f in PER_VARIANT_FIELDS]+GENERAL_FIELDS
    data=dict(zip(names,values)); required=[k for k in names if not k.endswith('variant_notes') and k!='overall_notes']
    if any(not str(data[k]).strip() for k in required): raise gr.Error('Complete all required fields.')
    existing=response_frame(); prior=existing[existing.page_key==page_key]; now=datetime.now(timezone.utc).isoformat()
    row={'review_order':int(PUBLIC[page_key]['review_order']),'page_key':page_key,'reviewer_code':REVIEWER_CODE,'review_revision':REVIEW_REVISION,'started_utc':prior.iloc[0].started_utc if not prior.empty else now,'submitted_utc':now,'assignment_commitment':PUBLIC[page_key]['assignment_commitment'],**data}
    digest=save_response(row); choices=pending(); nxt=choices[0] if choices else None
    return gr.update(choices=choices,value=nxt),f'Saved {page_key} | {digest[:12]}… | {status_text()}'

with gr.Blocks(title='Notebook 72R v2') as app:
    gr.Markdown('# Notebook 72R v2 — randomised A/B/C review\nJudge A, B and C independently. Preference is recorded separately and does not substitute for playability, print safety or annotation-removal judgements.')
    with gr.Row():
        page=gr.Dropdown(choices=pending(),value=(pending()[0] if pending() else None),label='Review item')
        status=gr.Textbox(value=status_text(),label='Status',interactive=False)
    with gr.Row(): imgA=gr.Image(label='A',height=650); imgB=gr.Image(label='B',height=650); imgC=gr.Image(label='C',height=650)
    components=[]
    with gr.Tabs():
        for v in VARIANTS:
            with gr.Tab(f'Judge {v}'):
                playable=gr.Radio(['YES','YES_WITH_DIFFICULTY','NO','UNCERTAIN'],value='UNCERTAIN',label=f'{v}: playable from?')
                damage=gr.Radio(['NONE','MINOR','MATERIAL','DESTROYED','UNCERTAIN'],value='UNCERTAIN',label=f'{v}: printed notation damage')
                removal=gr.Radio(['COMPLETE','MOSTLY_REMOVED','PARTIAL','NONE','NOT_APPLICABLE','UNCERTAIN'],value='PARTIAL',label=f'{v}: annotation removed?')
                residue=gr.Radio(['NO_VISIBLE_RESIDUE','MINOR_RESIDUE','VISIBLE_RESIDUE','OBJECTIONABLE_RESIDUE','NOT_APPLICABLE','UNCERTAIN'],value='VISIBLE_RESIDUE',label=f'{v}: visible annotation/residue')
                note=gr.Textbox(label=f'{v}: notes',lines=2)
                components += [playable,damage,removal,residue,note]
    pref=gr.Dropdown(['A>B>C','A>C>B','B>A>C','B>C>A','C>A>B','C>B>A','A=B>C','A=C>B','B=C>A','A>B=C','B>A=C','C>A=B','A=B=C'],value='A=B=C',label='Preference order only, ties allowed')
    conf=gr.Radio(['HIGH','MEDIUM','LOW'],value='MEDIUM',label='Confidence'); overall=gr.Textbox(label='Overall notes',lines=3); save=gr.Button('Save and load next',variant='primary')
    outputs=[imgA,imgB,imgC,status]+components+[pref,conf,overall]
    page.change(load_page,page,outputs); app.load(load_page,page,outputs)
    save.click(submit,[page]+components+[pref,conf,overall],[page,status]).then(load_page,page,outputs)
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 7. Close the review lock after 151 responses

In [ ]:
responses=response_frame(); require(len(responses)==len(public),f'Incomplete: {len(responses)} / {len(public)}')
require(set(responses.page_key)==set(public.page_key),'Response roster mismatch.')
for _,r in responses.iterrows():
    d=r.to_dict(); require(str(r.response_sha256)==response_hash(d),f'{r.page_key}: response hash mismatch.'); require(r.assignment_commitment==PUBLIC[r.page_key]['assignment_commitment'],f'{r.page_key}: commitment mismatch.')
payload={'prediction_lock_id':lock['lock_id'],'reviewer_code':REVIEWER_CODE,'review_revision':REVIEW_REVISION,'responses':len(responses),'responses_sha256':sha256_file(RESPONSES),'history_sha256':sha256_file(HISTORY),'review_key_sha256':sha256_file(SEALED/'review_key.csv')}
atomic_json(REVIEW_LOCK,{'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED','review_lock_id':object_hash(payload),'payload':payload})
print('REVIEW LOCK CLOSED:',json.loads(REVIEW_LOCK.read_text())['review_lock_id'])

## 8. Decode and create one row per page and actual variant

In [ ]:
require(review_closed(),'Close review lock first.')
seal=json.loads((SEALED/'review_key_seal.json').read_text()); require(seal['review_key_sha256']==sha256_file(SEALED/'review_key.csv'),'Review key changed.'); secret=base64.b64decode(seal['secret_b64']); key=pd.read_csv(SEALED/'review_key.csv'); key['review_order']=pd.to_numeric(key.review_order,errors='raise').astype('Int64')
for _,r in key.iterrows():
    base={k:(int(r[k]) if k=='review_order' else str(r[k])) for k in ['review_order','page_key','page','population','A','B','C']}; expected=hmac.new(secret,json.dumps(base,sort_keys=True).encode(),hashlib.sha256).hexdigest(); require(expected==r.assignment_commitment,f'{r.page_key}: key commitment failed.')
responses=response_frame(); wide=responses.merge(key,on=['review_order','page_key','assignment_commitment'],validate='one_to_one')
long=[]
for _,r in wide.iterrows():
    for label in VARIANTS:
        long.append({'review_order':int(r.review_order),'page_key':r.page_key,'page':r.page,'population':r.population,'concealed_label':label,'variant':r[label],'playable':r[f'{label}_playable'],'print_damage':r[f'{label}_print_damage'],'annotation_removal':r[f'{label}_annotation_removal'],'residue':r[f'{label}_residue'],'variant_notes':r[f'{label}_variant_notes'],'preference_order':r.preference_order,'preference_rank':next(i for i,g in enumerate(r.preference_order.split('>'),1) if label in g.split('=')),'confidence':r.confidence,'overall_notes':r.overall_notes})
long=pd.DataFrame(long); long.to_csv(DECODED/'variant_level_reviews.csv',index=False); wide.to_csv(DECODED/'decoded_page_reviews_wide.csv',index=False)
print(long.groupby(['variant','playable']).size())

## 9. Final comparison and exception tables

In [ ]:
# Variant-level outcomes are primary. Preference is secondary.
def crosstab_measure(measure):
    q=long.groupby(['variant',measure],as_index=False).size().rename(columns={'size':'pages'}); q['fraction_within_variant']=q.pages/q.groupby('variant').pages.transform('sum'); q.insert(0,'measure',measure); q=q.rename(columns={measure:'level'}); return q
summary=pd.concat([crosstab_measure(x) for x in ['playable','print_damage','annotation_removal','residue']],ignore_index=True); summary.to_csv(DECODED/'variant_outcome_summary.csv',index=False)
preference=(long.groupby(['variant','preference_rank'],as_index=False).size().rename(columns={'size':'pages'})); preference.to_csv(DECODED/'preference_rank_summary.csv',index=False)
by_source=(long.groupby(['population','variant'],as_index=False).agg(pages=('page_key','nunique'),playable_yes=('playable',lambda x:int((x=='YES').sum())),playable_no=('playable',lambda x:int((x=='NO').sum())),material_or_destroyed=('print_damage',lambda x:int(x.isin(['MATERIAL','DESTROYED']).sum())),annotation_complete_or_mostly=('annotation_removal',lambda x:int(x.isin(['COMPLETE','MOSTLY_REMOVED']).sum())),annotation_partial_or_none=('annotation_removal',lambda x:int(x.isin(['PARTIAL','NONE']).sum())),objectionable_residue=('residue',lambda x:int((x=='OBJECTIONABLE_RESIDUE').sum()))))
by_source.to_csv(DECODED/'results_by_source_group_and_variant.csv',index=False)
exceptions=long[(long.playable.isin(['NO','UNCERTAIN']))|long.print_damage.isin(['MATERIAL','DESTROYED','UNCERTAIN'])|long.annotation_removal.isin(['PARTIAL','NONE','UNCERTAIN'])|long.residue.isin(['OBJECTIONABLE_RESIDUE','UNCERTAIN'])].copy(); exceptions.to_csv(DECODED/'failure_and_exception_register.csv',index=False)

v88=long[long.variant=='CORRECTED_V88']; v78=long[long.variant=='BASELINE_V78']; observed=long[long.variant=='OBSERVED']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':'NOTEBOOK72R_V2_REVIEW_COMPLETE_EVIDENCE_REPORTED','promotion_authority':False,'prediction_lock_id':lock['lock_id'],'review_lock_id':json.loads(REVIEW_LOCK.read_text())['review_lock_id'],'pages':len(wide),'primary_interpretation':'Each variant was judged independently. Preference order is secondary and does not override playability, print-damage or annotation-removal outcomes.','CORRECTED_V88':{'playable_yes':int((v88.playable=='YES').sum()),'playable_no':int((v88.playable=='NO').sum()),'material_or_destroyed':int(v88.print_damage.isin(['MATERIAL','DESTROYED']).sum()),'annotation_complete_or_mostly':int(v88.annotation_removal.isin(['COMPLETE','MOSTLY_REMOVED']).sum()),'annotation_partial_or_none':int(v88.annotation_removal.isin(['PARTIAL','NONE']).sum()),'objectionable_residue':int((v88.residue=='OBJECTIONABLE_RESIDUE').sum())},'BASELINE_V78':{'playable_yes':int((v78.playable=='YES').sum()),'playable_no':int((v78.playable=='NO').sum()),'material_or_destroyed':int(v78.print_damage.isin(['MATERIAL','DESTROYED']).sum()),'annotation_complete_or_mostly':int(v78.annotation_removal.isin(['COMPLETE','MOSTLY_REMOVED']).sum())},'OBSERVED':{'playable_yes':int((observed.playable=='YES').sum()),'playable_no':int((observed.playable=='NO').sum())},'preference_first_place':long[long.preference_rank==1].variant.value_counts().to_dict(),'exception_rows':len(exceptions),'decision_note':'Do not infer a production pass from preference. Print destruction is a structural failure signal; incomplete removal is a separate cosmetic/usefulness outcome.'}
atomic_json(DECODED/'final_notebook72_verdict.json',record)
lines=['NOTEBOOK 72R V2 — RANDOMISED THREE-WAY REVIEW','',f'Pages: {len(wide)}',f'Prediction lock: {lock["lock_id"]}',f'Review lock: {record["review_lock_id"]}','','V8.8 independent outcomes',f'Playable YES / NO: {record["CORRECTED_V88"]["playable_yes"]} / {record["CORRECTED_V88"]["playable_no"]}',f'Material or destroyed print: {record["CORRECTED_V88"]["material_or_destroyed"]}',f'Annotation complete or mostly removed: {record["CORRECTED_V88"]["annotation_complete_or_mostly"]}',f'Annotation partial or none removed: {record["CORRECTED_V88"]["annotation_partial_or_none"]}',f'Objectionable residue: {record["CORRECTED_V88"]["objectionable_residue"]}','','Preference is secondary and does not substitute for the three independent outcome axes.','No automatic production promotion.']
(DECODED/'final_executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines))

## 10. Curated final package

In [ ]:
required=[OUT/'review_lock.json',INPUTS/'source_identity.json',REVIEW/'review_queue_public.csv',REVIEW/'responses.csv',REVIEW/'response_history.jsonl',SEALED/'review_key.csv',SEALED/'review_key_seal.json',DECODED/'variant_level_reviews.csv',DECODED/'decoded_page_reviews_wide.csv',DECODED/'variant_outcome_summary.csv',DECODED/'preference_rank_summary.csv',DECODED/'results_by_source_group_and_variant.csv',DECODED/'failure_and_exception_register.csv',DECODED/'final_notebook72_verdict.json',DECODED/'final_executive_summary.txt']
for p in required: require(p.exists(),f'Missing: {p}')
manifest=[]
for src in required:
    dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 72R v2 final randomised three-way review package. Variant-level safety and usability judgements are primary; preference is secondary. No production promotion is granted.\n')
with zipfile.ZipFile(UPLOAD/'notebook72r_v2_final_review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip': z.write(p,p.name)
print('Final bundle:',UPLOAD/'notebook72r_v2_final_review_bundle.zip')

## Interpretation

The review deliberately keeps three questions separate:

- **Playability** is the practical page-level outcome.
- **Print damage** is the structural-safety outcome.
- **Annotation removal/residue** is the cosmetic-usefulness outcome.

A version can therefore be preferred but still unplayable, playable but cosmetically poor, or cosmetically clean but structurally unsafe. Preference must never erase those distinctions.